# PERG-HGP — validation Blackwell 3D, 30 millions de points

Ce notebook teste le backend expérimental `perg_hgp.backends.power_cover_3d_cuda` sur un accélérateur NVIDIA Blackwell (cible Colab G4 / RTX PRO 6000 Blackwell, classe 96 Go). Il est conçu pour être lancé avec **Exécution > Tout exécuter**, après avoir sélectionné le GPU adapté. Les calculs et artefacts restent sur le disque local `/content` : Google Drive/FUSE n'est jamais utilisé comme espace de calcul.

## Limites à lire avant l'exécution

- Le backend GPU et son noyau CUDA `cubical_msf_gpu` n'ont pas été exécutés dans le dépôt CPU qui a produit ce notebook : cette exécution Blackwell constitue précisément leur validation matérielle.
- `K=10` est une résolution finie fixée ; ce réglage n'est pas une affirmation de consistance asymptotique.
- La sortie enveloppée est une hiérarchie cubique H0 intérieure/extérieure. Des labels plats 30 M ne sont ni sélectionnés ni garantis.
- La régularisation utilise un budget local `s_i <= min(B_abs, gamma R_K(x_i))`; si `gamma < 1/2`, le rapport publie le contrat multiplicatif observé sur le nuage quantifié.
- La grille uniforme anisotrope choisit `r_min` à partir du quantile local pilote de `R_K`, puis le relève explicitement si le budget de cellules l'impose. Le quantile cible et le plan réellement abordable sont tous deux publiés ; ce n'est pas encore un octree adaptatif.
- L'exactitude de RAPIDS RBC est auditée **empiriquement sur des requêtes échantillonnées** contre cuVS brute force. Cet audit n'est pas une preuve portant sur les 30 millions de requêtes.
- Le garde global des requêtes de puissance est strict sous l'enveloppe float32 déclarée : une requête incertaine provoque un arrêt, sans repli ANN silencieux. Cette enveloppe n'est pas une preuve par arithmétique dirigée.
- `CONDITIONAL_PASS` signifie que les contrats techniques conditionnels à RBC sont satisfaits et qu'au moins une persistance synthétique dépasse deux fois la borne complète. `INCONCLUSIVE` signale notamment une résolution locale insuffisante ; `FAIL` signale la violation d'un contrat dur.


In [ ]:
# Paramètres principaux — modifier ici seulement avant une exécution complète.
from pathlib import Path

REPO_URL = "https://github.com/Ludwig-H/E-HGP.git"
GIT_REF = "main"
GIT_COMMIT = ""  # @param {type:"string"}; renseigner le commit publié pour un run figé
RAPIDS_CSP_URL = "https://github.com/rapidsai/rapidsai-csp-utils.git"
RAPIDS_CSP_REF = "main"
RAPIDS_CSP_COMMIT = "36a2d5696695680364dbe2b0bb44104acd7f19fe"
N_FULL = 30_000_000
K = 10
M_REG = 64
REGULARIZATION_MODE = "local_distortion"
DISTORTION_GAMMA = 0.08         # s_i <= gamma R_K(x_i), gamma < 1/2
DISTORTION_ABSOLUTE_CAP = 0.25  # unités du nuage
MIN_RESOLVED_RADIUS_OVERRIDE = 0.0  # 0 = quantile pilote + plancher imposé par MAX_GRID_CELLS
LOCAL_SCALE_TARGET_QUANTILE = 0.05  # vise au plus 5 % de R_K sous r_min avant contrainte de grille
LOCAL_SCALE_DIMENSION = 3.0         # extrapolation pilote N^(-1/d), hypothèse 3D explicitement publiée
MAX_RELATIVE_SPATIAL_ERROR = 0.25  # 2H <= eta r_min, hors delta_num publié
MAX_GRID_CELLS = 50_000_000     # refus explicite, jamais de coarsening silencieux
CANDIDATE_K_INITIAL = 64
RBC_MAX_QUERY_K = 1_024          # frontière du kernel RBC de la pile RAPIDS épinglée
CANDIDATE_K_MAX = 1_023  # RBC accepte 1 024 rangs, dont un réservé au garde global
MIN_FREE_VRAM_GIB = 65.0
MIN_DISK_FREE_GIB = 30.0
MAX_VRAM_FRACTION = 0.85
MAX_HOST_RAM_FRACTION = 0.85
RUN_FULL_30M = True
FULL_REPEATS = 1
verbose = True  # @param {type:"boolean"}

# Progression facultative. Elle exécute trois graines à chaque petite échelle.
RUN_PROGRESSION = False
PROGRESSION_SIZES = (100_000, 1_000_000, 3_000_000, 10_000_000)
SMALL_SEEDS = (20260710, 20260711, 20260712)
FULL_SEED = 20260710

# Le pilote ne sélectionne plus un kappa global : il audite l'échelle R_K.
# Le solveur maximise localement l'entropie sous les deux budgets ci-dessus.
PILOT_N = 250_000
PILOT_SAMPLE_SIZE = 8_192
DATA_CHUNK_SIZE = 1_000_000
COMPUTE_CHUNK_SIZE = 262_144
POWER_QUERY_CHUNK_SIZE = 65_536
NEIGHBOR_AUDIT_QUERIES = 32
MONITOR_INTERVAL_S = 0.5

REPO_DIR = Path("/content/E-HGP-perg-hgp-blackwell")
PACKAGE_DIR = REPO_DIR / "perg_hgp"
ARTIFACT_ROOT = Path("/content/perg_hgp_blackwell_results")

assert CANDIDATE_K_INITIAL == 64 and CANDIDATE_K_MAX + 1 == RBC_MAX_QUERY_K
assert K == 10 and M_REG == 64 and N_FULL == 30_000_000
assert FULL_REPEATS >= 1
assert REGULARIZATION_MODE == "local_distortion"
assert 0.0 < DISTORTION_GAMMA < 0.5
assert DISTORTION_ABSOLUTE_CAP >= 0.0 and MIN_RESOLVED_RADIUS_OVERRIDE >= 0.0
assert 0.0 < LOCAL_SCALE_TARGET_QUANTILE < 0.5 and LOCAL_SCALE_DIMENSION > 0.0
assert 0.0 < MAX_RELATIVE_SPATIAL_ERROR <= 1.0 and MAX_GRID_CELLS > 0
assert type(verbose) is bool

def vprint(*args, **kwargs):
    if verbose:
        kwargs.setdefault("flush", True)
        print(*args, **kwargs)

print({"grid_policy": "pilot_quantile_then_cell_budget", "r_min_override": MIN_RESOLVED_RADIUS_OVERRIDE, "target_quantile": LOCAL_SCALE_TARGET_QUANTILE, "eta": MAX_RELATIVE_SPATIAL_ERROR, "max_cells": MAX_GRID_CELLS, "run_full": RUN_FULL_30M, "full_repeats": FULL_REPEATS, "verbose": verbose})


## 1. Prévol matériel bloquant

Le notebook refuse explicitement un runtime sans GPU, une compute capability antérieure à Blackwell (`major < 10`) ou moins de 65 Gio de VRAM actuellement libre. Le manifeste brut est conservé pour la reproductibilité.


In [ ]:
import json, math, os, platform, shutil, subprocess, sys, time

def capture(command):
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=30)
    return {"command": command, "returncode": result.returncode, "output": result.stdout.strip()}

nvidia_smi_full = capture(["nvidia-smi"])
nvidia_smi_query = capture([
    "nvidia-smi",
    "--query-gpu=name,driver_version,memory.total,memory.free,compute_cap",
    "--format=csv,noheader,nounits",
])
if nvidia_smi_full["returncode"] != 0:
    raise RuntimeError("Aucun GPU NVIDIA utilisable. Sélectionner un runtime Colab G4/Blackwell puis recommencer.")

try:
    import psutil
    vm = psutil.virtual_memory()
    ram = {"total_bytes": int(vm.total), "available_bytes": int(vm.available)}
except Exception:
    page = os.sysconf("SC_PAGE_SIZE")
    ram = {"total_bytes": int(page * os.sysconf("SC_PHYS_PAGES"))}

disk = shutil.disk_usage("/content")
try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("PyTorch ne voit aucun périphérique CUDA.")
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    free_bytes, total_bytes = (int(v) for v in torch.cuda.mem_get_info(0))
    gpu_name = torch.cuda.get_device_name(0)
except Exception as exc:
    raise RuntimeError("Impossible d'établir la compute capability et la VRAM CUDA avant installation RAPIDS.") from exc

HARDWARE_MANIFEST_PRE = {
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python": sys.version,
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "cpu_model": platform.processor(),
    "ram": ram,
    "disk_content": {"total_bytes": disk.total, "free_bytes": disk.free},
    "gpu_name": gpu_name,
    "compute_capability": [cc_major, cc_minor],
    "vram_total_bytes": total_bytes,
    "vram_free_bytes": free_bytes,
    "nvidia_smi": nvidia_smi_full,
    "nvidia_smi_query": nvidia_smi_query,
}
print(json.dumps(HARDWARE_MANIFEST_PRE, indent=2, ensure_ascii=False))

if cc_major < 10:
    raise RuntimeError(f"GPU refusé : compute capability {cc_major}.{cc_minor}, Blackwell >= 10.x exigé.")
free_gib = free_bytes / 2**30
if free_gib < MIN_FREE_VRAM_GIB:
    raise MemoryError(f"GPU refusé : {free_gib:.2f} Gio libres, minimum requis {MIN_FREE_VRAM_GIB:.2f} Gio.")
disk_free_gib = disk.free / 2**30
if disk_free_gib < MIN_DISK_FREE_GIB:
    raise OSError(f"Scratch local insuffisant : {disk_free_gib:.2f} Gio libres, minimum {MIN_DISK_FREE_GIB:.2f} Gio.")
print(f"Prévol accepté : {gpu_name}, CC {cc_major}.{cc_minor}, {free_gib:.2f} Gio VRAM et {disk_free_gib:.2f} Gio disque libres.")


## 2. Installation officielle RAPIDS Colab

La cellule suivante utilise une révision épinglée du script officiel `rapidsai-csp-utils/colab/pip-install.py`. Toute installation modifie des modules binaires potentiellement déjà chargés : le notebook s'arrête donc **inconditionnellement** juste après l'installation. Choisir **Exécution > Redémarrer la session**, puis relancer depuis les paramètres ; la seconde exécution détecte la pile déjà installée, vérifie ses imports et configure RMM avant toute allocation CuPy.


In [ ]:
import importlib.util

required_gpu_modules = ("cupy", "cuml", "cuvs", "rmm")
missing_gpu_modules = [name for name in required_gpu_modules if importlib.util.find_spec(name) is None]
if missing_gpu_modules:
    utils_dir = Path("/content/rapidsai-csp-utils")
    if utils_dir.exists() and not (utils_dir / ".git").is_dir():
        raise RuntimeError(f"{utils_dir} existe mais n'est pas un clone Git ; le déplacer avant de continuer.")
    if not utils_dir.exists():
        subprocess.run(["git", "clone", "--no-checkout", RAPIDS_CSP_URL, str(utils_dir)], check=True)
    subprocess.run(["git", "-C", str(utils_dir), "remote", "set-url", "origin", RAPIDS_CSP_URL], check=True)
    subprocess.run(["git", "-C", str(utils_dir), "fetch", "--prune", "origin", RAPIDS_CSP_REF], check=True)
    subprocess.run(["git", "-C", str(utils_dir), "checkout", "--detach", RAPIDS_CSP_COMMIT], check=True)
    installer = utils_dir / "colab" / "pip-install.py"
    subprocess.run([sys.executable, str(installer)], check=True)
    print("Installation RAPIDS terminée : redémarrage obligatoire avant tout import binaire.")
    raise RuntimeError("Redémarrage de session obligatoire après modification de la pile RAPIDS ; relancer ensuite depuis les paramètres.")
else:
    print("CuPy, cuML et cuVS sont déjà importables ; installation RAPIDS ignorée.")

# Vérifier les imports binaires, puis unifier CuPy et cuML sur RMM avant toute allocation CuPy.
try:
    import cupy as cp
    import cuml, cuvs, rmm
except Exception as exc:
    raise RuntimeError("Pile RAPIDS incomplète ou incompatible ; redémarrer la session après l'installation, puis relancer depuis les paramètres.") from exc
from rmm.allocators.cupy import rmm_cupy_allocator
cupy_pool = cp.get_default_memory_pool()
if cupy_pool.used_bytes() or cupy_pool.total_bytes():
    raise RuntimeError("Le pool CuPy est déjà actif : redémarrer la session avant de configurer RMM.")
cp.cuda.set_allocator(rmm_cupy_allocator)
RMM_ALLOCATOR_STATUS = {"allocator": "rmm_cupy_allocator", "cupy_pool_initially_empty": True}
RAPIDS_IMPORT_VERSIONS = {name: getattr(module, "__version__", "unknown") for name, module in (("cupy", cp), ("cuml", cuml), ("cuvs", cuvs), ("rmm", rmm))}
print({"rapids": RAPIDS_IMPORT_VERSIONS, "allocator": RMM_ALLOCATOR_STATUS})


In [ ]:
# Récupère la branche publiée, puis détache le commit fusionné pour une installation reproductible.
if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"{REPO_DIR} existe mais n'est pas un clone Git ; le déplacer avant de continuer.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", GIT_REF], check=True)
checkout_target = GIT_COMMIT.strip() or "FETCH_HEAD"
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", checkout_target], check=True)
NOTEBOOK_RUNTIME_REQUIREMENTS = ("numba", "psutil")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PACKAGE_DIR) + "[test]", *NOTEBOOK_RUNTIME_REQUIREMENTS], check=True)
# Un noyau déjà démarré ne relit pas automatiquement le .pth créé par une installation éditable.
import importlib
PACKAGE_IMPORT_ROOT = str(PACKAGE_DIR.resolve())
sys.path[:] = [entry for entry in sys.path if entry != PACKAGE_IMPORT_ROOT]
sys.path.insert(0, PACKAGE_IMPORT_ROOT)
importlib.invalidate_caches()
import perg_hgp as perg_hgp_package
PACKAGE_IMPORTED_FROM = Path(perg_hgp_package.__file__).resolve()
EXPECTED_PACKAGE_INIT = (PACKAGE_DIR / "perg_hgp" / "__init__.py").resolve()
if PACKAGE_IMPORTED_FROM != EXPECTED_PACKAGE_INIT:
    raise RuntimeError(f"perg_hgp importé depuis {PACKAGE_IMPORTED_FROM}, attendu {EXPECTED_PACKAGE_INIT} ; redémarrer la session.")
import numba, psutil
GIT_HEAD = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if GIT_COMMIT.strip() and GIT_HEAD != GIT_COMMIT.strip():
    raise RuntimeError(f"Commit installé inattendu : {GIT_HEAD} au lieu de {GIT_COMMIT}.")
print({"git_ref_fetched": GIT_REF, "git_head": GIT_HEAD, "package_dir": str(PACKAGE_DIR), "imported_from": str(PACKAGE_IMPORTED_FROM), "numba": numba.__version__, "psutil": psutil.__version__})


## 3. Tous les tests CPU

Toute la suite `perg_hgp/tests` est exécutée avec CUDA masqué dans un sous-processus isolé des plugins et options pytest externes au projet. Le test qui vérifie l'absence optionnelle de CuPy est ignoré puisque CuPy est installé ; tous les chemins CPU restent exécutés. Les versions et la sortie complète sont affichées, avec une relance détaillée au premier échec. Le benchmark GPU ne démarre pas si pytest échoue.


In [ ]:
from importlib import metadata as importlib_metadata

CPU_TESTS_OK = False
cpu_test_env = os.environ.copy()
ignored_pytest_env = [name for name in ("PYTEST_ADDOPTS", "PYTEST_PLUGINS") if cpu_test_env.pop(name, None) is not None]
cpu_test_env.update({
    "CUDA_VISIBLE_DEVICES": "",
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
    "PYTHONHASHSEED": "0",
})
cpu_test_versions = {}
for distribution in ("numpy", "scipy", "scikit-learn", "torch", "pytest", "cupy-cuda12x", "cuml-cu12", "cuvs-cu12", "rmm-cu12"):
    try:
        cpu_test_versions[distribution] = importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        pass
cpu_test_cwd = PACKAGE_DIR.parent.parent.resolve()
cpu_test_target = str((PACKAGE_DIR / "tests").resolve())
cpu_test_command = [sys.executable, "-m", "pytest", "-q", "-ra", "--tb=short", cpu_test_target]
print(json.dumps({
    "command": cpu_test_command, "cwd": str(cpu_test_cwd),
    "python": sys.version, "versions": cpu_test_versions,
    "ignored_external_pytest_environment": ignored_pytest_env,
}, indent=2, ensure_ascii=False))
cpu_test_result = subprocess.run(
    cpu_test_command, cwd=cpu_test_cwd, env=cpu_test_env,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(cpu_test_result.stdout, end="" if cpu_test_result.stdout.endswith("\n") else "\n")
if cpu_test_result.returncode != 0:
    pip_check_result = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(f"\npip check (code {pip_check_result.returncode}) :\n{pip_check_result.stdout}")
    diagnostic_command = [sys.executable, "-m", "pytest", "-vv", "-x", "--tb=long", cpu_test_target]
    print("\nRelance diagnostique jusqu'au premier échec :", diagnostic_command)
    diagnostic_result = subprocess.run(
        diagnostic_command, cwd=cpu_test_cwd, env=cpu_test_env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(diagnostic_result.stdout, end="" if diagnostic_result.stdout.endswith("\n") else "\n")
    raise RuntimeError(f"Suite CPU échouée (code {cpu_test_result.returncode}) ; voir le diagnostic pytest complet ci-dessus.")
CPU_TESTS_OK = True
print("Suite CPU complète : OK")


In [ ]:
# Imports GPU seulement après la suite CPU.
import cupy as cp
import cuml, cuvs
import numpy as np

from perg_hgp.backends.power_cover_3d_cuda import (
    CertifiedPowerKNN,
    PowerCover3D,
    PowerCoverConfig,
    brute_force_power_topk,
    cubical_msf_cpu,
    cubical_msf_gpu,
    estimate_memory,
    solve_entropy_gibbs_budget,
)
from perg_hgp.backends.power_cover_3d_cuda.cuda_runtime import RBCAuditedIndex, collect_hardware_manifest
from perg_hgp.backends.power_cover_3d_cuda.spatial_core import _cuda_regularize_budget_chunk

properties = cp.cuda.runtime.getDeviceProperties(0)
cc = (int(properties["major"]), int(properties["minor"]))
free_bytes, total_bytes = (int(v) for v in cp.cuda.runtime.memGetInfo())
if cc[0] < 10 or free_bytes / 2**30 < MIN_FREE_VRAM_GIB:
    raise RuntimeError("Le contrôle CUDA post-installation ne satisfait plus le contrat Blackwell/VRAM.")
GPU_HARDWARE_MANIFEST = collect_hardware_manifest()
GPU_HARDWARE_MANIFEST.update({"cuvs": getattr(cuvs, "__version__", "unknown"), "git_head": GIT_HEAD})
print(json.dumps(GPU_HARDWARE_MANIFEST, indent=2, ensure_ascii=False))
GPU_STACK_OK = True


## 4. Audit empirique RAPIDS RBC contre cuVS brute force

Les distances triées sont comparées, plutôt que les identifiants, car les doublons et ex aequo peuvent avoir plusieurs représentants exacts. Les cas dégénérés sont intentionnels : uniforme, distribution asymétrique, plan, ligne, nuage constant et doublons. Le dernier cas ajoute des poids de puissance variables et compare aussi l'oracle de puissance certifié à l'oracle CPU bloqué.


In [ ]:
def make_rbc_case(name, n=4_096, seed=123):
    rng = cp.random.RandomState(seed)
    if name == "uniforme":
        points = rng.random_sample((n, 3)).astype(cp.float32)
    elif name == "skew":
        u = rng.random_sample((n, 3)).astype(cp.float32)
        points = cp.column_stack((cp.exp(5.0 * (u[:, 0] - 1.0)), u[:, 1] ** 4, 0.02 * u[:, 2]))
    elif name == "plan":
        xy = rng.standard_normal((n, 2)).astype(cp.float32)
        points = cp.column_stack((xy, cp.zeros(n, dtype=cp.float32)))
    elif name == "ligne":
        x = cp.linspace(-1.0, 1.0, n, dtype=cp.float32)
        points = cp.column_stack((x, cp.zeros(n, dtype=cp.float32), cp.zeros(n, dtype=cp.float32)))
    elif name == "constant":
        points = cp.zeros((n, 3), dtype=cp.float32)
    elif name == "doublons":
        base = rng.standard_normal((n // 4, 3)).astype(cp.float32)
        points = cp.repeat(base, 4, axis=0)
    else:
        raise ValueError(name)
    points = cp.ascontiguousarray(points, dtype=cp.float32)
    ids = cp.linspace(0, n - 1, 64, dtype=cp.int64)
    queries = points[ids].copy()
    if name not in {"constant", "doublons"}:
        queries += 1e-3 * rng.standard_normal(queries.shape).astype(cp.float32)
    return points, cp.ascontiguousarray(queries)

RBC_AUDIT_RESULTS = {}
rbc_cases = ("uniforme", "skew", "plan", "ligne", "constant", "doublons")
for case_id, case_name in enumerate(rbc_cases, start=1):
    case_started = time.perf_counter()
    vprint(f"[audit RBC {case_id}/8] cas {case_name} : démarrage")
    try:
        case_points, case_queries = make_rbc_case(case_name)
        index = RBCAuditedIndex(case_points, max_k=K)
        audit = index.audit_against_bruteforce(case_queries, k=K).to_dict()
        audit["passed"] = bool(audit["values_match"])
        RBC_AUDIT_RESULTS[case_name] = audit
    except Exception as exc:
        RBC_AUDIT_RESULTS[case_name] = {"passed": False, "error": repr(exc)}
    finally:
        for variable in ("index", "case_points", "case_queries"):
            if variable in locals():
                del locals()[variable]
        cp.get_default_memory_pool().free_all_blocks()
        vprint(f"[audit RBC {case_id}/8] cas {case_name} : terminé en {time.perf_counter() - case_started:.1f} s")

# Poids de puissance variables : audit Euclidien RBC + garde globale de puissance.
weighted_started = time.perf_counter()
vprint("[audit RBC 7/8] poids dynamiques : démarrage")
try:
    weighted_points, weighted_queries = make_rbc_case("uniforme", n=65_536, seed=999)
    smoke_candidate_max = min(CANDIDATE_K_MAX, 255)
    weighted_index = RBCAuditedIndex(weighted_points, max_k=smoke_candidate_max + 1)
    euclidean_audit = weighted_index.audit_against_bruteforce(weighted_queries, k=K).to_dict()
    ids = cp.arange(weighted_points.shape[0], dtype=cp.float32)
    dynamic_weights = (0.02 * cp.sin(ids * 0.017) ** 2).astype(cp.float32)
    oracle = CertifiedPowerKNN(
        weighted_points, dynamic_weights, weighted_index, K=K,
        candidate_k_initial=CANDIDATE_K_INITIAL, candidate_k_max=smoke_candidate_max, strict=True,
    )
    power_result = oracle.query(weighted_queries[:48])
    exact_values, _ = brute_force_power_topk(
        cp.asnumpy(weighted_queries[:48]), cp.asnumpy(weighted_points), cp.asnumpy(dynamic_weights), K, site_block=512
    )
    power_error = float(np.max(np.abs(cp.asnumpy(power_result.power_values) - exact_values[:, -1])))
    power_scale = max(1.0, float(np.max(np.abs(exact_values[:, -1]))))
    power_tolerance = float(256.0 * np.finfo(np.float32).eps * power_scale)
    RBC_AUDIT_RESULTS["poids_dynamiques"] = {
        "euclidean_audit": euclidean_audit,
        "power_max_abs_error": power_error,
        "power_tolerance": power_tolerance,
        "certified_fraction": float(power_result.diagnostics.certified_fraction),
        "uncertain_queries": int(power_result.diagnostics.uncertain_queries),
        "passed": bool(euclidean_audit["values_match"] and power_error <= power_tolerance and power_result.diagnostics.uncertain_queries == 0),
    }
except Exception as exc:
    RBC_AUDIT_RESULTS["poids_dynamiques"] = {"passed": False, "error": repr(exc)}
finally:
    for variable in ("weighted_points", "weighted_queries", "weighted_index", "dynamic_weights", "oracle", "power_result"):
        if variable in locals():
            del locals()[variable]
    cp.get_default_memory_pool().free_all_blocks()
    vprint(f"[audit RBC 7/8] poids dynamiques : terminé en {time.perf_counter() - weighted_started:.1f} s")

# Frontière réelle du kernel RBC : 1 023 candidats plus le rang de garde 1 024.
boundary_started = time.perf_counter()
vprint("[audit RBC 8/8] frontière k=1024 : démarrage")
try:
    boundary_points, boundary_queries = make_rbc_case("uniforme", n=RBC_MAX_QUERY_K**2, seed=1000)
    boundary_index = RBCAuditedIndex(boundary_points, max_k=RBC_MAX_QUERY_K)
    boundary_audit = boundary_index.audit_against_bruteforce(
        boundary_queries[:NEIGHBOR_AUDIT_QUERIES], k=RBC_MAX_QUERY_K
    ).to_dict()
    boundary_audit["passed"] = bool(boundary_audit["values_match"])
    RBC_AUDIT_RESULTS["frontiere_rbc_k1024"] = boundary_audit
except Exception as exc:
    RBC_AUDIT_RESULTS["frontiere_rbc_k1024"] = {"passed": False, "error": repr(exc)}
finally:
    for variable in ("boundary_points", "boundary_queries", "boundary_index"):
        if variable in locals():
            del locals()[variable]
    cp.get_default_memory_pool().free_all_blocks()
    vprint(f"[audit RBC 8/8] frontière k=1024 : terminé en {time.perf_counter() - boundary_started:.1f} s")

# Ex aequo au-delà du cap : certifier l'intervalle de valeur, pas l'identité des sites.
try:
    tie_points, tie_queries = make_rbc_case("constant", n=4_096, seed=1001)
    tie_index = RBCAuditedIndex(tie_points, max_k=64)
    tie_index.audit_against_bruteforce(tie_queries[:8], k=K)
    tie_oracle = CertifiedPowerKNN(tie_points, cp.zeros(4_096, dtype=cp.float32), tie_index, K=K, candidate_k_initial=32, candidate_k_max=63, strict=True)
    tie_result = tie_oracle.query(tie_queries[:8])
    RBC_AUDIT_RESULTS["power_ties_value_interval"] = {
        "conditional_guard_fraction": float(tie_result.diagnostics.conditional_guard_pass_fraction),
        "radius_error_max": float(tie_result.diagnostics.radius_error_max),
        "histogram": tie_result.diagnostics.candidate_histogram,
        "passed": bool(tie_result.diagnostics.uncertain_queries == 0 and cp.all(tie_result.radii == 0).item()),
    }
except Exception as exc:
    RBC_AUDIT_RESULTS["power_ties_value_interval"] = {"passed": False, "error": repr(exc)}
finally:
    for variable in ("tie_points", "tie_queries", "tie_index", "tie_oracle", "tie_result"):
        if variable in locals():
            del locals()[variable]
    cp.get_default_memory_pool().free_all_blocks()

print(json.dumps(RBC_AUDIT_RESULTS, indent=2, ensure_ascii=False, allow_nan=False))
RBC_AUDITS_OK = all(row.get("passed", False) for row in RBC_AUDIT_RESULTS.values())
assert RBC_AUDITS_OK, "Au moins un audit RBC/cuVS a échoué : le run strict 30M est bloqué."


## 5. Compilation NVRTC et parité des noyaux CUDA

La première invocation force la compilation du `RawModule` NVRTC contenant les six noyaux Borůvka. Pour chaque champ, les composantes GPU sont comparées à la référence CPU à tous les niveaux d'activation distincts. Le second smoke compile le `RawKernel` de régularisation locale et compare directement `z`, `a`, `s`, l'entropie et la perplexité à l'oracle CPU sur les régimes budget nul, actif et uniforme, avec doublons, supports non puissances de deux et support maximal 1 024. Un smoke end-to-end RBC/grille complète ensuite cette parité algorithmique.


In [ ]:
rng_cpu = np.random.default_rng(20260710)
msf_fields = {
    "negatifs_ex_aequo": rng_cpu.integers(-4, 5, size=(8, 7, 6)).astype(np.float32),
    "motif_ex_aequo": (np.indices((7, 6, 5)).sum(axis=0) % 4 - 2).astype(np.float32),
    "constant_negatif": np.full((5, 4, 3), -1.5, dtype=np.float32),
}
MSF_SMOKE_RESULTS = {}
for field_id, (field_name, field_cpu) in enumerate(msf_fields.items(), start=1):
    vprint(f"[smoke MSF {field_id}/{len(msf_fields)}] {field_name} : démarrage")
    cpu_result = cubical_msf_cpu(field_cpu)
    cp.cuda.Device().synchronize()
    started = time.perf_counter()
    gpu_result = cubical_msf_gpu(cp.asarray(field_cpu))
    cp.cuda.Device().synchronize()
    elapsed = time.perf_counter() - started
    levels = np.unique(field_cpu)
    thresholds = [np.nextafter(levels[0], -np.inf, dtype=np.float32), *levels.tolist(), np.nextafter(levels[-1], np.inf, dtype=np.float32)]
    partitions_match = all(
        np.array_equal(cpu_result.components_at_cut(float(level)), gpu_result.components_at_cut(float(level)))
        for level in thresholds
    )
    MSF_SMOKE_RESULTS[field_name] = {
        "n_vertices": int(field_cpu.size),
        "gpu_seconds_including_first_compile_if_applicable": elapsed,
        "partitions_match_all_thresholds": bool(partitions_match),
        "passed": bool(partitions_match and gpu_result.edges.shape == (field_cpu.size - 1, 2)),
    }
    vprint(f"[smoke MSF {field_id}/{len(msf_fields)}] {field_name} : terminé en {elapsed:.2f} s")
print(json.dumps(MSF_SMOKE_RESULTS, indent=2))
MSF_SMOKE_OK = all(row["passed"] for row in MSF_SMOKE_RESULTS.values())
assert MSF_SMOKE_OK, "Le MSF cubique CUDA ne reproduit pas la filtration CPU."

# Parité directe du RawKernel local fusionné contre l'oracle CPU.
def fused_budget_parity_case(name, support, duplicates=False):
    support = int(support)
    axis = np.linspace(0.0, 0.9, support, dtype=np.float32)
    points_cpu = np.column_stack((axis, 0.11 * np.sin(3.0 * axis), 0.07 * np.cos(5.0 * axis))).astype(np.float32)
    points_cpu[0] = 0.0
    if duplicates:
        points_cpu[: min(4, support)] = 0.0
    queries_cpu = np.zeros((3, 3), dtype=np.float32)
    one_row_distances = np.linalg.norm(points_cpu, axis=1).astype(np.float32)
    distances_cpu = np.broadcast_to(one_row_distances, (3, support)).copy()
    indices_cpu = np.broadcast_to(np.arange(support, dtype=np.int64), (3, support)).copy()
    uniform_radius = float(np.sqrt(np.mean(one_row_distances * one_row_distances)))
    budgets_cpu = np.asarray((0.0, 0.55 * uniform_radius, 1.05 * uniform_radius), dtype=np.float32)
    gpu_outputs = _cuda_regularize_budget_chunk(
        cp.asarray(queries_cpu), cp.asarray(points_cpu), cp.asarray(distances_cpu),
        cp.asarray(indices_cpu), cp.asarray(budgets_cpu),
    )
    cp.cuda.Device().synchronize()
    centers_gpu, additive_gpu, distortion_gpu, entropy_gpu, kappa_gpu = (cp.asnumpy(value) for value in gpu_outputs)
    costs_cpu = distances_cpu * distances_cpu
    oracle = solve_entropy_gibbs_budget(costs_cpu, budgets_cpu * budgets_cpu, max_iter=28)
    probabilities = np.asarray(oracle.probabilities, dtype=np.float32)
    neighbors = points_cpu[indices_cpu]
    centers_cpu = np.sum(probabilities[:, :, None] * neighbors, axis=1)
    offsets = neighbors - centers_cpu[:, None, :]
    additive_cpu = np.sum(probabilities * np.sum(offsets * offsets, axis=2), axis=1)
    distortion_cpu = np.sqrt(np.maximum(np.sum((queries_cpu - centers_cpu) ** 2, axis=1) + additive_cpu, 0.0))
    entropy_cpu = np.asarray(oracle.entropy, dtype=np.float32)
    kappa_cpu = np.asarray(oracle.effective_kappa, dtype=np.float32)
    errors = {
        "centers_max_abs": float(np.max(np.abs(centers_gpu - centers_cpu))),
        "additive_max_abs": float(np.max(np.abs(additive_gpu - additive_cpu))),
        "distortion_max_abs": float(np.max(np.abs(distortion_gpu - distortion_cpu))),
        "entropy_max_abs": float(np.max(np.abs(entropy_gpu - entropy_cpu))),
        "kappa_max_relative": float(np.max(np.abs(kappa_gpu - kappa_cpu) / np.maximum(np.abs(kappa_cpu), 1.0))),
        "budget_violation_max": float(np.max(np.maximum(distortion_gpu - budgets_cpu, 0.0))),
    }
    nontrivial_active_solution = bool(kappa_gpu[1] > (4.0 if duplicates else 1.01) and np.linalg.norm(centers_gpu[1]) > 0.0)
    uniform_solution = bool(abs(float(kappa_gpu[2]) - support) <= 3e-3 * max(1, support))
    passed = bool(
        errors["centers_max_abs"] <= 5e-4
        and errors["additive_max_abs"] <= 5e-4
        and errors["distortion_max_abs"] <= 5e-4
        and errors["entropy_max_abs"] <= 3e-3
        and errors["kappa_max_relative"] <= 3e-3
        and errors["budget_violation_max"] <= 5e-4
        and nontrivial_active_solution and uniform_solution
    )
    return {
        "name": name, "support": support, "duplicates": bool(duplicates),
        "budgets": budgets_cpu.tolist(), "kappa_gpu": kappa_gpu.tolist(),
        "kappa_cpu": kappa_cpu.tolist(), "errors": errors,
        "nontrivial_active_solution": nontrivial_active_solution,
        "uniform_solution": uniform_solution, "passed": passed,
    }

FUSED_BUDGET_PARITY = {
    name: fused_budget_parity_case(name, support, duplicates)
    for name, support, duplicates in (
        ("non_power_7", 7, False),
        ("duplicates_non_power_13", 13, True),
        ("boundary_1024", 1_024, False),
    )
}
FUSED_BUDGET_PARITY_OK = all(row["passed"] for row in FUSED_BUDGET_PARITY.values())
print(json.dumps({"fused_budget_cpu_parity": FUSED_BUDGET_PARITY}, indent=2, allow_nan=False))
assert FUSED_BUDGET_PARITY_OK, "Le RawKernel local fusionné diverge de l'oracle CPU."
cp.get_default_memory_pool().free_all_blocks()

# Smoke end-to-end du solveur local fusionné avec RBC et grille.
local_smoke_points, _ = make_rbc_case("uniforme", n=4_096, seed=20260711)
local_smoke_config = PowerCoverConfig(
    K=10, kappa=1.0, m_reg=64, grid_shape=(1, 1, 1),
    regularization_mode="local_distortion", distortion_budget_absolute=0.25,
    distortion_budget_relative=0.08, local_scale_k=10,
    min_resolved_radius=0.5, max_relative_spatial_error=0.5, max_grid_cells=500_000,
    chunk_size=2_048, power_chunk_size=4_096, candidate_k_initial=32, candidate_k_max=63,
    neighbor_audit_queries=16, require_neighbor_audit=True, max_vram_fraction=MAX_VRAM_FRACTION,
)
local_smoke = PowerCover3D(local_smoke_config, backend="cuda").fit(local_smoke_points)
LOCAL_BUDGET_SMOKE = {
    "solver_backend": local_smoke.sites.diagnostics.solver_backend,
    "gamma_observed": local_smoke.sites.diagnostics.relative_distortion_ratio_max,
    "budget_violation_max": local_smoke.sites.diagnostics.distortion_budget_violation_max,
    "conditional_guard_fraction": local_smoke.field.diagnostics.conditional_guard_pass_fraction,
    "uncertain_queries": local_smoke.field.diagnostics.uncertain_queries,
}
LOCAL_BUDGET_SMOKE_OK = bool(
    LOCAL_BUDGET_SMOKE["solver_backend"] == "cuda_fused_budget_v1"
    and LOCAL_BUDGET_SMOKE["gamma_observed"] is not None
    and LOCAL_BUDGET_SMOKE["gamma_observed"] < 0.5
    and LOCAL_BUDGET_SMOKE["budget_violation_max"] <= 5e-4
    and LOCAL_BUDGET_SMOKE["conditional_guard_fraction"] == 1.0
    and LOCAL_BUDGET_SMOKE["uncertain_queries"] == 0
    and FUSED_BUDGET_PARITY_OK
)
print(json.dumps({"local_budget_cuda_smoke": LOCAL_BUDGET_SMOKE}, indent=2))
assert LOCAL_BUDGET_SMOKE_OK, "Le solveur local CUDA fusionné a échoué son smoke end-to-end."
del local_smoke, local_smoke_points
cp.get_default_memory_pool().free_all_blocks()


## 6. Générateur 3D en chunks et pilote d'échelle locale

Le générateur préalloue uniquement `N x 3` en `float32` puis remplit le tableau par blocs. Le pilote estime le quantile 5 % de `R_K` et l'extrapole à la taille du run par la loi `N^(-1/d)` avec `d=3`, hypothèse enregistrée et non présentée comme une preuve pour des données arbitraires. Pour chaque nuage, le code calcule ensuite le plus petit `r_min` compatible avec `MAX_GRID_CELLS` sur sa boîte réelle : `r_min` est le maximum du quantile local projeté et de ce plancher de faisabilité. Si le plancher de grille domine et laisse plus de 5 % des échelles locales sous `r_min`, le verdict reste explicitement `INCONCLUSIVE` ; aucun fallback global de `kappa` n'est exécuté.


In [ ]:
def generate_mixture_3d(n, seed, chunk_size=DATA_CHUNK_SIZE, verbose=verbose):
    n = int(n)
    rng = cp.random.RandomState(int(seed))
    means = cp.asarray([
        [-0.75, -0.55, -0.35], [0.65, -0.45, 0.20], [-0.25, 0.70, 0.45],
        [0.65, 0.55, -0.55], [0.00, 0.00, 0.00],
    ], dtype=cp.float32)
    scales = cp.asarray([
        [0.11, 0.07, 0.16], [0.09, 0.15, 0.08], [0.17, 0.08, 0.10],
        [0.08, 0.09, 0.14], [0.28, 0.12, 0.20],
    ], dtype=cp.float32)
    points = cp.empty((n, 3), dtype=cp.float32)
    started = time.perf_counter()
    report_every = max(1, int(np.ceil(n / chunk_size / 10)))
    for block_id, start in enumerate(range(0, n, chunk_size)):
        stop = min(n, start + chunk_size)
        count = stop - start
        components = rng.randint(0, means.shape[0], size=count).astype(cp.int32, copy=False)
        block = rng.standard_normal((count, 3)).astype(cp.float32)
        block *= scales[components]
        block += means[components]
        points[start:stop] = block
        del components, block
        if verbose and (block_id % report_every == 0 or stop == n):
            cp.cuda.Device().synchronize()
            elapsed = time.perf_counter() - started
            rate = stop / max(elapsed, 1e-9)
            eta = (n - stop) / max(rate, 1e-9)
            print(
                f"[génération] {stop:,}/{n:,} points ({100.0 * stop / n:.1f} %) | "
                f"{rate:,.0f} points/s | écoulé {elapsed:.1f} s | ETA {eta:.1f} s",
                flush=True,
            )
    cp.cuda.Device().synchronize()
    expected_bytes = n * 3 * np.dtype(np.float32).itemsize
    assert int(points.nbytes) == expected_bytes
    if verbose:
        print(f"[génération] Nuage N×3 prêt : {points.nbytes / 2**30:.3f} Gio, {time.perf_counter() - started:.2f} s", flush=True)
    return cp.ascontiguousarray(points)

pilot_started = time.perf_counter()
vprint(f"[pilote échelle 1/4] génération de {PILOT_N:,} points")
pilot_points = generate_mixture_3d(PILOT_N, FULL_SEED, verbose=verbose)
vprint("[pilote échelle 2/4] construction de l'index RBC")
pilot_index = RBCAuditedIndex(pilot_points, max_k=M_REG)
vprint(f"[pilote échelle 3/4] audit de {NEIGHBOR_AUDIT_QUERIES} requêtes contre cuVS brute force")
pilot_audit_ids = cp.linspace(0, PILOT_N - 1, NEIGHBOR_AUDIT_QUERIES, dtype=cp.int64)
PILOT_RBC_AUDIT = pilot_index.audit_against_bruteforce(pilot_points[pilot_audit_ids], k=M_REG).to_dict()
PILOT_RBC_AUDIT["passed"] = bool(PILOT_RBC_AUDIT["values_match"])
RBC_AUDIT_RESULTS["pilote_echelle"] = PILOT_RBC_AUDIT
RBC_AUDITS_OK = bool(RBC_AUDITS_OK and PILOT_RBC_AUDIT["passed"])
assert RBC_AUDITS_OK, "L'index RBC du pilote d'échelle ne concorde pas avec cuVS."
vprint("[pilote échelle 4/4] quantiles de R_K sur l'échantillon")
pilot_sample_ids = cp.linspace(0, PILOT_N - 1, min(PILOT_SAMPLE_SIZE, PILOT_N), dtype=cp.int64)
pilot_distances, _ = pilot_index.query(pilot_points[pilot_sample_ids], K)
pilot_rk = pilot_distances[:, K - 1]
PILOT_QUANTILE_LEVELS = (0.0, LOCAL_SCALE_TARGET_QUANTILE, 0.5, 0.9, 0.99, 1.0)
pilot_quantile_levels = cp.asarray(PILOT_QUANTILE_LEVELS, dtype=cp.float32)
pilot_quantile_values = cp.quantile(pilot_rk, pilot_quantile_levels)
PILOT_RK_QUANTILES = {f"{level:g}": float(value) for level, value in zip(PILOT_QUANTILE_LEVELS, cp.asnumpy(pilot_quantile_values))}
pilot_target_key = f"{LOCAL_SCALE_TARGET_QUANTILE:g}"
PILOT_LOCAL_RADIUS_TARGET_FULL = PILOT_RK_QUANTILES[pilot_target_key] * (PILOT_N / N_FULL) ** (1.0 / LOCAL_SCALE_DIMENSION)
if MIN_RESOLVED_RADIUS_OVERRIDE == 0.0 and PILOT_LOCAL_RADIUS_TARGET_FULL <= 0.0:
    raise RuntimeError("Le quantile pilote local est nul ; fournir MIN_RESOLVED_RADIUS_OVERRIDE pour déclarer une échelle positive.")

def _planned_grid_from_spans(spans, radius):
    spans = np.asarray(spans, dtype=np.float64)
    diameter_budget = (MAX_RELATIVE_SPATIAL_ERROR * float(radius)) ** 2
    unresolved = {axis for axis in range(3) if spans[axis] > 0.0}
    one_cell = {axis for axis in range(3) if spans[axis] == 0.0}
    fixed_squared = 0.0
    width_limit = np.inf
    while unresolved:
        remaining_budget = diameter_budget - fixed_squared
        if remaining_budget <= 0.0:
            raise RuntimeError("Les axes à une cellule épuisent le budget diagonal local.")
        width_limit = np.sqrt(remaining_budget / len(unresolved))
        newly_fixed = {axis for axis in unresolved if spans[axis] <= width_limit}
        if not newly_fixed:
            break
        one_cell.update(newly_fixed)
        fixed_squared = float(sum(spans[axis] ** 2 for axis in one_cell))
        unresolved.difference_update(newly_fixed)
    if not unresolved:
        shape = (1, 1, 1)
    else:
        shape = tuple(1 if axis in one_cell else max(1, int(np.ceil(spans[axis] / width_limit))) for axis in range(3))
    return shape, int(shape[0]) * int(shape[1]) * int(shape[2])

def choose_run_min_resolved_radius(points, n_points):
    lower = cp.asnumpy(cp.min(points, axis=0)).astype(np.float64, copy=False)
    upper = cp.asnumpy(cp.max(points, axis=0)).astype(np.float64, copy=False)
    spans = np.maximum(upper - lower, 0.0)
    pilot_projected = PILOT_RK_QUANTILES[pilot_target_key] * (PILOT_N / int(n_points)) ** (1.0 / LOCAL_SCALE_DIMENSION)
    desired = float(MIN_RESOLVED_RADIUS_OVERRIDE or pilot_projected)
    if not np.isfinite(desired) or desired <= 0.0:
        raise RuntimeError("La sélection pilote de r_min n'est pas strictement positive.")
    desired_shape, desired_cells = _planned_grid_from_spans(spans, desired)
    chosen = desired
    reason = "manual_override" if MIN_RESOLVED_RADIUS_OVERRIDE > 0.0 else "pilot_quantile"
    if desired_cells > MAX_GRID_CELLS:
        active_axes = max(1, int(np.count_nonzero(spans > 0.0)))
        low = desired
        high = max(desired, float(np.max(spans)) * np.sqrt(active_axes) / MAX_RELATIVE_SPATIAL_ERROR)
        while _planned_grid_from_spans(spans, high)[1] > MAX_GRID_CELLS:
            high *= 2.0
        for _ in range(64):
            middle = 0.5 * (low + high)
            if _planned_grid_from_spans(spans, middle)[1] <= MAX_GRID_CELLS:
                high = middle
            else:
                low = middle
        chosen = float(np.nextafter(high * (1.0 + 64.0 * np.finfo(np.float32).eps), np.inf))
        reason += "+grid_budget_floor"
    chosen_shape, chosen_cells = _planned_grid_from_spans(spans, chosen)
    if chosen_cells > MAX_GRID_CELLS:
        raise RuntimeError("La sélection de r_min n'a pas fermé le budget de grille.")
    return chosen, {
        "selection_reason": reason, "target_quantile": LOCAL_SCALE_TARGET_QUANTILE,
        "dimension_assumption": LOCAL_SCALE_DIMENSION, "pilot_projected_radius": float(pilot_projected),
        "manual_override": float(MIN_RESOLVED_RADIUS_OVERRIDE), "desired_radius": float(desired),
        "selected_radius": float(chosen), "desired_shape": list(desired_shape),
        "desired_cells": int(desired_cells), "selected_shape": list(chosen_shape),
        "selected_cells": int(chosen_cells), "bbox_spans": spans.tolist(),
    }
print(json.dumps({
    "K": K, "m_reg": M_REG, "regularization_mode": REGULARIZATION_MODE,
    "distortion_gamma": DISTORTION_GAMMA, "absolute_cap": DISTORTION_ABSOLUTE_CAP,
    "R_K_quantiles": PILOT_RK_QUANTILES, "target_quantile": LOCAL_SCALE_TARGET_QUANTILE,
    "projected_target_radius_at_30M": PILOT_LOCAL_RADIUS_TARGET_FULL,
    "neighbor_audit": PILOT_RBC_AUDIT,
}, indent=2))
vprint(f"[pilote échelle] terminé en {time.perf_counter() - pilot_started:.1f} s")
del pilot_audit_ids, pilot_sample_ids, pilot_distances, pilot_rk, pilot_quantile_levels, pilot_quantile_values, pilot_index, pilot_points
cp.get_default_memory_pool().free_all_blocks()


## 7. Instrumentation stricte et sauvegarde locale

Le moniteur utilise NVML — ou `nvidia-smi` en repli — comme mesure VRAM autoritative, puis échantillonne le RSS hôte et l'espace disque. Les compteurs du pool CuPy historique sont conservés à titre diagnostique mais restent normalement nuls après le routage vers RMM. `PowerCover3D` synchronise déjà chaque étage ; le wrapper synchronise aussi immédiatement avant et après `fit`. Les temps par étage proviennent du backend, et le temps mural est mesuré séparément. Avec le `@param` Colab `verbose=True`, les six étages affichent leur avancement, leur débit et leur ETA ; un heartbeat toutes les 30 secondes indique aussi la VRAM et le RSS pendant les opérations opaques de cuML.


In [ ]:
import datetime as dt
import gc
import re
import threading
import traceback
from numba import njit

STAGE_LABELS = {
    "input": "préparation et normalisation",
    "raw_index": "index RBC brut et audit",
    "regularization": "régularisation entropique",
    "power_index": "index RBC des sites et audit",
    "power_field": "champ de puissance cubique",
    "cubical_msf": "MSF cubique Borůvka",
}
OPERATION_LABELS = {
    "build_rapids_rbc": "construction RAPIDS RBC",
    "audit_against_cuvs_bruteforce": "audit cuVS brute force",
    "audit_grid_against_cuvs_bruteforce": "audit des centres de grille contre cuVS",
    "device_to_host_and_deterministic_sort": "copie GPU→CPU et tri déterministe du MSF",
}
UNIT_LABELS = {"points": "points", "cubes": "cubes", "edges": "arêtes"}

def format_duration(seconds):
    seconds = max(0, int(round(float(seconds))))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours:d} h {minutes:02d} min {seconds:02d} s"
    if minutes:
        return f"{minutes:d} min {seconds:02d} s"
    return f"{seconds:d} s"

class NotebookProgress:
    def __init__(self, enabled=True, percent_step=5.0):
        self.enabled = bool(enabled)
        self.percent_step = float(percent_step)
        self.stage_order = {name: i + 1 for i, name in enumerate(STAGE_LABELS)}
        self.last_percent = {}
        self.last_print = {}
        self.current_label = None
        self.current_started = None

    def _prefix(self, stage):
        position = self.stage_order.get(stage)
        return f"[PERG-HGP {position}/{len(self.stage_order)}]" if position else "[PERG-HGP]"

    def __call__(self, event):
        if not self.enabled:
            return
        stage = str(event["stage"])
        kind = str(event["kind"])
        label = STAGE_LABELS.get(stage, stage)
        prefix = self._prefix(stage)
        details = event.get("details", {})
        elapsed = float(event.get("elapsed_seconds", 0.0))
        if kind == "start":
            self.current_label = label
            self.current_started = time.monotonic()
            self.last_percent[stage] = -self.percent_step
            self.last_print[stage] = time.monotonic()
            unit = event.get("unit")
            scope = "" if unit == "stage" else f" — {int(event.get('total', 0)):,} {UNIT_LABELS.get(unit, unit)}"
            print(f"{prefix} {label} : démarrage{scope}", flush=True)
            return
        if kind == "message":
            operation = OPERATION_LABELS.get(details.get("operation"), details.get("operation", "travail en cours"))
            print(f"{prefix} {label} : {operation} — écoulé {format_duration(elapsed)}", flush=True)
            return
        if kind == "progress":
            completed = int(event["completed"])
            total = int(event["total"])
            percent = 100.0 if total == 0 else 100.0 * completed / total
            now = time.monotonic()
            should_print = (
                completed == total
                or percent - self.last_percent.get(stage, -self.percent_step) >= self.percent_step
                or now - self.last_print.get(stage, 0.0) >= 30.0
                or "phase" in details
            )
            if not should_print:
                return
            self.last_percent[stage] = percent
            self.last_print[stage] = now
            unit = UNIT_LABELS.get(event.get("unit"), event.get("unit", "éléments"))
            rate = completed / max(elapsed, 1e-9)
            eta = (total - completed) / max(rate, 1e-9)
            marker = ""
            if "chunk" in details:
                marker = f" — chunk {int(details['chunk'])}/{int(details['chunks'])}"
            elif "phase" in details:
                marker = f" — phase {int(details['phase'])}/{int(details['max_phases'])}"
            certification = ""
            if "certified" in details:
                certification = f" — certifiés {int(details['certified']):,}, incertains {int(details['uncertain']):,}"
            print(
                f"{prefix} {label} : {completed:,}/{total:,} {unit} ({percent:.1f} %){marker}{certification} | "
                f"{rate:,.0f} {unit}/s | écoulé {format_duration(elapsed)} | ETA {format_duration(eta)}",
                flush=True,
            )
            return
        if kind == "done":
            print(f"{prefix} {label} : terminé en {format_duration(elapsed)}", flush=True)

    def activity(self, label):
        if not self.enabled:
            return
        self.current_label = str(label)
        self.current_started = time.monotonic()
        print(f"[PERG-HGP] {self.current_label} : démarrage", flush=True)

    def heartbeat(self, sampler):
        if not self.enabled or self.current_label is None or self.current_started is None:
            return
        elapsed = time.monotonic() - self.current_started
        used = getattr(sampler, "last_nvml_used", 0) / 2**30
        total = max(1, getattr(sampler, "nvml_total", 0)) / 2**30
        rss = getattr(sampler, "last_rss", 0) / 2**30
        print(
            f"[PERG-HGP heartbeat] {self.current_label} — écoulé {format_duration(elapsed)} — "
            f"VRAM {used:.1f}/{total:.1f} Gio — RSS {rss:.1f} Gio",
            flush=True,
        )

def assert_local_scratch(path):
    resolved = str(Path(path).resolve())
    if resolved.startswith("/content/drive") or "/gdrive/" in resolved.lower():
        raise RuntimeError("Calcul ou sauvegarde sur Google Drive/FUSE interdits. Utiliser /content.")
    return Path(resolved)

ARTIFACT_ROOT = assert_local_scratch(ARTIFACT_ROOT)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

class ResourceSampler:
    def __init__(self, interval_s=0.5, progress=None, heartbeat_s=30.0):
        self.interval_s = float(interval_s)
        self.progress = progress
        self.heartbeat_s = float(heartbeat_s)
        self.last_heartbeat = time.monotonic()
        self.stop_event = threading.Event()
        self.thread = None
        self.samples = 0
        self.peak_allocated = 0
        self.peak_reserved = 0
        self.peak_nvml_used = 0
        self.nvml_total = 0
        self.last_nvml_used = 0
        self.peak_rss = 0
        self.last_rss = 0
        self.min_disk_free = shutil.disk_usage("/content").free
        self.errors = []
        self.nvml_source = "unavailable"
        self._nvml = None
        self._nvml_handle = None
        try:
            import pynvml
            pynvml.nvmlInit()
            self._nvml = pynvml
            self._nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            self.nvml_source = "pynvml"
        except Exception as exc:
            self.errors.append("pynvml: " + repr(exc))
            if shutil.which("nvidia-smi"):
                self.nvml_source = "nvidia-smi polling"

    def _gpu_used(self):
        if self._nvml is not None:
            info = self._nvml.nvmlDeviceGetMemoryInfo(self._nvml_handle)
            return int(info.used), int(info.total)
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=5, check=True,
        )
        used_mib, total_mib = (float(part.strip()) for part in result.stdout.splitlines()[0].split(","))
        return int(used_mib * 2**20), int(total_mib * 2**20)

    def sample(self):
        try:
            pool = cp.get_default_memory_pool()
            self.peak_allocated = max(self.peak_allocated, int(pool.used_bytes()))
            self.peak_reserved = max(self.peak_reserved, int(pool.total_bytes()))
            used, total = self._gpu_used()
            self.last_nvml_used = used
            self.peak_nvml_used = max(self.peak_nvml_used, used)
            self.nvml_total = max(self.nvml_total, total)
            try:
                self.last_rss = int(psutil.Process().memory_info().rss)
                self.peak_rss = max(self.peak_rss, self.last_rss)
            except Exception:
                pass
            self.min_disk_free = min(self.min_disk_free, int(shutil.disk_usage("/content").free))
            self.samples += 1
        except Exception as exc:
            if len(self.errors) < 20:
                self.errors.append(repr(exc))

    def _loop(self):
        while not self.stop_event.wait(self.interval_s):
            self.sample()
            now = time.monotonic()
            if self.progress is not None and now - self.last_heartbeat >= self.heartbeat_s:
                try:
                    self.progress.heartbeat(self)
                except Exception as exc:
                    if len(self.errors) < 20:
                        self.errors.append("heartbeat: " + repr(exc))
                self.last_heartbeat = now

    def start(self):
        self.sample()
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()
        return self

    def stop(self):
        self.stop_event.set()
        if self.thread is not None:
            self.thread.join(timeout=max(5.0, 2.0 * self.interval_s))
        self.sample()
        denominator = max(1, self.nvml_total)
        return {
            "samples": self.samples, "interval_seconds": self.interval_s,
            "legacy_cupy_pool_allocated_peak_bytes": self.peak_allocated,
            "legacy_cupy_pool_reserved_peak_bytes": self.peak_reserved,
            "nvml_used_peak_bytes": self.peak_nvml_used, "nvml_total_bytes": self.nvml_total,
            "vram_peak_fraction": self.peak_nvml_used / denominator,
            "rss_peak_bytes": self.peak_rss, "disk_free_min_bytes": self.min_disk_free,
            "nvml_source": self.nvml_source, "monitor_errors": self.errors,
        }

@njit(cache=False)
def h0_persistences_numba(births, edges, weights):
    n = births.size
    parent = np.arange(n, dtype=np.int32)
    component_birth = births.copy()
    persistence = np.empty(edges.shape[0], dtype=np.float32)
    for edge_id in range(edges.shape[0]):
        u = int(edges[edge_id, 0])
        v = int(edges[edge_id, 1])
        ru = u
        while parent[ru] != ru:
            ru = int(parent[ru])
        cursor = u
        while parent[cursor] != cursor:
            following = int(parent[cursor])
            parent[cursor] = ru
            cursor = following
        rv = v
        while parent[rv] != rv:
            rv = int(parent[rv])
        cursor = v
        while parent[cursor] != cursor:
            following = int(parent[cursor])
            parent[cursor] = rv
            cursor = following
        bu = component_birth[ru]
        bv = component_birth[rv]
        if bu < bv or (bu == bv and ru < rv):
            survivor, loser = ru, rv
        else:
            survivor, loser = rv, ru
        value = weights[edge_id] - component_birth[loser]
        persistence[edge_id] = value if value > 0.0 else 0.0
        parent[loser] = survivor
        component_birth[survivor] = min(bu, bv)
    return persistence

def persistence_summary(forest, comparison_threshold):
    started = time.perf_counter()
    persistence = h0_persistences_numba(
        np.asarray(forest.births, dtype=np.float32),
        np.asarray(forest.edges, dtype=np.int32),
        np.asarray(forest.weights, dtype=np.float32),
    )
    quantiles = np.quantile(persistence, (0.5, 0.9, 0.99, 0.999, 1.0)) if persistence.size else np.zeros(5)
    result = {
        "finite_count": int(persistence.size), "positive_count": int(np.count_nonzero(persistence > 0)),
        "q50": float(quantiles[0]), "q90": float(quantiles[1]), "q99": float(quantiles[2]),
        "q999": float(quantiles[3]), "max": float(quantiles[4]),
        "comparison_threshold": float(comparison_threshold),
        "count_above_comparison_threshold": int(np.count_nonzero(persistence > comparison_threshold)),
        "cpu_seconds": time.perf_counter() - started,
    }
    del persistence
    return result

def json_safe_failure_value(value):
    if isinstance(value, dict):
        return {str(key): json_safe_failure_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe_failure_value(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_safe_failure_value(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value) if np.isfinite(value) else repr(float(value))
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, float) and not np.isfinite(value):
        return repr(value)
    return value

RUN_RECORDS = []
FULL_RUN_RECORDS = []

def run_power_cover(points, n_points, seed, repeat, scale_kind):
    # Réserver un rang de garde sous les deux limites RBC : sqrt(N) et kernel k<=1024.
    run_candidate_max = min(CANDIDATE_K_MAX, RBC_MAX_QUERY_K - 1, math.isqrt(int(n_points)) - 1)
    if run_candidate_max < CANDIDATE_K_INITIAL:
        raise ValueError(f"N={n_points} ne permet pas le plancher de candidats RBC.")
    run_min_resolved_radius, grid_selection = choose_run_min_resolved_radius(points, n_points)
    print("Sélection locale/grille :", json.dumps(grid_selection, indent=2))
    config = PowerCoverConfig(
        K=K, kappa=1.0, m_reg=M_REG, grid_shape=(1, 1, 1),
        regularization_mode=REGULARIZATION_MODE,
        distortion_budget_absolute=DISTORTION_ABSOLUTE_CAP,
        distortion_budget_relative=DISTORTION_GAMMA, local_scale_k=K,
        min_resolved_radius=run_min_resolved_radius,
        max_relative_spatial_error=MAX_RELATIVE_SPATIAL_ERROR,
        max_grid_cells=MAX_GRID_CELLS,
        chunk_size=COMPUTE_CHUNK_SIZE, power_chunk_size=POWER_QUERY_CHUNK_SIZE,
        candidate_k_initial=CANDIDATE_K_INITIAL, candidate_k_max=run_candidate_max,
        strict_certification=True,
        neighbor_audit_queries=NEIGHBOR_AUDIT_QUERIES, require_neighbor_audit=True,
        max_vram_fraction=MAX_VRAM_FRACTION,
    )
    plan = estimate_memory(int(n_points), config, n_cubes=grid_selection["selected_cells"])
    print("Plan mémoire préliminaire avec grille sélectionnée (non mesuré) :", json.dumps(plan.to_dict(), indent=2))
    progress_reporter = NotebookProgress(enabled=verbose)
    sampler = ResourceSampler(
        MONITOR_INTERVAL_S, progress=progress_reporter if verbose else None
    ).start()
    hierarchy = None
    resources = None
    cp.cuda.Device().synchronize()
    wall_start = time.perf_counter()
    try:
        hierarchy = PowerCover3D(
            config, backend="cuda",
            progress=progress_reporter if verbose else None,
        ).fit(points)
        cp.cuda.Device().synchronize()
        wall_seconds = time.perf_counter() - wall_start

        h = float(hierarchy.field.spec.half_diagonal)
        delta_num = float(hierarchy.field.numerical_radius_error)
        s = float(hierarchy.sites.diagnostics.s_max)
        epsilon_x = float(hierarchy.report.input_quantization_radius)
        base_bound = epsilon_x + s + h + delta_num
        envelope_bound = epsilon_x + s + 2.0 * (h + delta_num)
        if not np.isclose(base_bound, hierarchy.report.base_total_interleaving_radius, rtol=2e-6, atol=2e-7):
            raise RuntimeError("La borne base recalculée ne concorde pas avec le rapport.")
        if not np.isclose(envelope_bound, hierarchy.report.total_interleaving_radius, rtol=2e-6, atol=2e-7):
            raise RuntimeError("La borne enveloppe recalculée ne concorde pas avec le rapport.")
        progress_reporter.activity("calcul des persistances H0 sur CPU")
        persistence = persistence_summary(hierarchy.base_forest, 2.0 * envelope_bound)
        persistence["max_over_bound"] = persistence["max"] / max(envelope_bound, np.finfo(float).tiny)
        vprint(f"[PERG-HGP] persistances H0 terminées en {persistence['cpu_seconds']:.1f} s")

        timestamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        run_name = f"{scale_kind}_n{int(n_points)}_seed{int(seed)}_rep{int(repeat)}_{timestamp}"
        run_dir = assert_local_scratch(ARTIFACT_ROOT / run_name)
        run_dir.mkdir(parents=True, exist_ok=False)
        progress_reporter.activity(f"sauvegarde des artefacts dans {run_dir}")
        hierarchy.save(str(run_dir), include_sites=False)
        np.savez_compressed(
            run_dir / "acceptance_inputs.npz",
            epsilon_X=np.float64(epsilon_x), s_up=np.float64(s), H=np.float64(h), delta_num=np.float64(delta_num),
            base_bound=np.float64(base_bound), envelope_bound=np.float64(envelope_bound), persistence_max=np.float64(persistence["max"]),
        )
        vprint("[PERG-HGP] artefacts principaux sauvegardés")
        resources = sampler.stop()
        attempt_wall_seconds = time.perf_counter() - wall_start
    except Exception as exc:
        if resources is None:
            resources = sampler.stop()
        partial_neighbor_audits = json_safe_failure_value(getattr(exc, "neighbor_audits", {}))
        if partial_neighbor_audits:
            print("Audits de voisinage au moment de l'échec :", json.dumps(partial_neighbor_audits, indent=2, ensure_ascii=False))
        print("Mesures au moment de l'échec :", json.dumps(resources, indent=2))
        failure_stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        failure_path = ARTIFACT_ROOT / f"failure_{scale_kind}_n{int(n_points)}_{failure_stamp}.json"
        with open(failure_path, "w", encoding="utf-8") as stream:
            json.dump({"error": repr(exc), "traceback": traceback.format_exc(), "config": config.to_dict(), "grid_selection": grid_selection, "resources": resources, "neighbor_audits": partial_neighbor_audits, "git_head": GIT_HEAD}, stream, indent=2, ensure_ascii=False, allow_nan=False)
            stream.write("\n")
        raise
    try:
        rss_after = int(psutil.Process().memory_info().rss)
    except Exception:
        rss_after = 0
    metrics = {
        "run_name": run_name, "run_dir": str(run_dir), "scale_kind": scale_kind,
        "n_points": int(n_points), "seed": int(seed), "repeat": int(repeat),
        "git_ref": GIT_REF, "git_head": GIT_HEAD, "config": config.to_dict(), "grid_selection": grid_selection,
        "wall_seconds": wall_seconds, "attempt_wall_seconds": attempt_wall_seconds,
        "stage_timings_seconds": hierarchy.timings,
        "resources": resources, "rss_after_save_bytes": rss_after,
        "disk_free_after_save_bytes": int(shutil.disk_usage("/content").free),
        "regularization": hierarchy.sites.diagnostics.to_dict(),
        "grid": hierarchy.field.spec.to_dict(),
        "power_queries": hierarchy.field.diagnostics.to_dict(),
        "accuracy": hierarchy.report.to_dict(), "memory_estimate": hierarchy.memory_estimate.to_dict(),
        "neighbor_audits": hierarchy.neighbor_audits, "persistence": persistence,
        "bound_terms": {"epsilon_X": epsilon_x, "s_up": s, "H": h, "delta_num": delta_num, "base_total": base_bound, "envelope_total": envelope_bound},
        "pilot": {
            "R_K_quantiles": PILOT_RK_QUANTILES,
            "target_quantile": LOCAL_SCALE_TARGET_QUANTILE,
            "dimension_assumption": LOCAL_SCALE_DIMENSION,
            "projected_target_radius_at_30M": PILOT_LOCAL_RADIUS_TARGET_FULL,
            "distortion_gamma": DISTORTION_GAMMA,
            "absolute_cap": DISTORTION_ABSOLUTE_CAP,
        },
        "preflight_manifest": HARDWARE_MANIFEST_PRE,
        "gpu_manifest": GPU_HARDWARE_MANIFEST, "allocator": RMM_ALLOCATOR_STATUS,
        "smoke_tests": {"cpu_tests_ok": CPU_TESTS_OK, "rbc": RBC_AUDIT_RESULTS, "cubical_msf": MSF_SMOKE_RESULTS, "fused_budget_cpu_parity": FUSED_BUDGET_PARITY, "local_budget_cuda": LOCAL_BUDGET_SMOKE},
    }
    with open(run_dir / "benchmark_metrics.json", "w", encoding="utf-8") as stream:
        json.dump(metrics, stream, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)
        stream.write("\n")
    print("Temps par étage :", json.dumps(hierarchy.timings, indent=2))
    print("Pics ressources :", json.dumps(resources, indent=2))
    print("Bornes et persistance :", json.dumps({"base": base_bound, "envelope": envelope_bound, "persistence": persistence}, indent=2))
    del hierarchy
    cp.get_default_memory_pool().free_all_blocks()
    return metrics



## 8. Progression facultative

Si `RUN_PROGRESSION=True`, cette cellule exécute successivement 100 k, 1 M, 3 M et 10 M points avec trois graines. Elle est désactivée par défaut afin que le protocole 30 M reste directement lançable. Chaque échelle est générée, résolue avec le même contrat local puis libérée séparément.


In [ ]:
if RUN_PROGRESSION:
    progression_total = len(PROGRESSION_SIZES) * len(SMALL_SEEDS)
    progression_run = 0
    for n_points in PROGRESSION_SIZES:
        for seed in SMALL_SEEDS:
            progression_run += 1
            print(f"\n=== Progression N={n_points:,}, seed={seed} ===")
            vprint(f"[progression {progression_run}/{progression_total}] lancement")
            progression_started = time.perf_counter()
            small_points = generate_mixture_3d(n_points, seed, verbose=verbose)
            record = run_power_cover(small_points, n_points, seed, 0, "progression")
            RUN_RECORDS.append(record)
            vprint(f"[progression {progression_run}/{progression_total}] terminée en {format_duration(time.perf_counter() - progression_started)}")
            del small_points
            cp.get_default_memory_pool().free_all_blocks()
else:
    print("Progression ignorée (RUN_PROGRESSION=False).")


## 9. Exécution cible 30 millions

Une seule répétition complète est effectuée par défaut. Augmenter `FULL_REPEATS` uniquement si le budget de temps le permet. Le solveur sous budget local effectue une seule régularisation fusionnée, sans relance globale de `kappa`. Aucune matrice dense `N x N`, `N x K` globale ni liste explicite des arêtes de grille n'est construite par ce notebook.


In [ ]:
if RUN_FULL_30M:
    print(f"Génération du mélange 3D complet : N={N_FULL:,}")
    full_started = time.perf_counter()
    full_points = generate_mixture_3d(N_FULL, FULL_SEED, verbose=verbose)
    vprint(f"[run 30 M] génération terminée en {format_duration(time.perf_counter() - full_started)}")
    for repeat in range(FULL_REPEATS):
        print(f"\n=== Run complet {repeat + 1}/{FULL_REPEATS} ===")
        repeat_started = time.perf_counter()
        record = run_power_cover(full_points, N_FULL, FULL_SEED, repeat, "full")
        FULL_RUN_RECORDS.append(record)
        RUN_RECORDS.append(record)
        vprint(f"[run 30 M {repeat + 1}/{FULL_REPEATS}] terminé en {format_duration(time.perf_counter() - repeat_started)}")
    del full_points
    cp.get_default_memory_pool().free_all_blocks()
    vprint(f"[run 30 M] durée totale {format_duration(time.perf_counter() - full_started)}")
else:
    print("Run 30 M ignoré explicitement (RUN_FULL_30M=False).")


## 10. Acceptation automatique

Contrats durs : parité du solveur CUDA fusionné avec l'oracle CPU, statut relatif exact, audits RBC brut et puissance attachés au run avec les capacités attendues, fraction de garde conditionnelle égale à 1, zéro requête incertaine, budget de distorsion borné, VRAM mesurée sous 85 %, sélection de `r_min` traçable sous le budget de grille, et cohérence des deux bornes `epsilon_X+s+(H+delta)` / `epsilon_X+s+2(H+delta)`. Le kernel actuel ne retourne pas de résidu de simplexe : le champ diagnostique est donc `null` et n'est pas utilisé comme preuve. La persistance H0 est calculée par règle de l'aîné. Si le plancher de grille place plus que le quantile cible de sites sous `r_min`, ou si aucune persistance ne dépasse `2 ×` la borne enveloppe, le résultat est `INCONCLUSIVE`. Un succès reste `CONDITIONAL_PASS` tant que RBC n'est audité que par échantillon.


In [ ]:
RESIDUAL_TOL = 5e-4
MAX_UNRESOLVED_LOCAL_SCALE_FRACTION = LOCAL_SCALE_TARGET_QUANTILE

def record_rbc_audits_pass(record):
    try:
        audits = record["neighbor_audits"]
        config = record["config"]
        n_points = int(record["n_points"])
        raw_values = audits["raw_rbc_vs_cuvs_bruteforce"]
        power_values = audits["power_grid_centers_rbc_vs_cuvs_bruteforce"]
        raw_status = audits["raw_neighbor_status"]
        power_status = audits["power_neighbor_status"]
        raw_expected_k = min(n_points, max(int(config["m_reg"]), int(config["local_scale_k"])))
        power_expected_k = min(n_points, int(config["candidate_k_max"]) + 1)
        return bool(
            raw_values["values_match"] and power_values["values_match"]
            and int(raw_values["queries"]) > 0 and int(power_values["queries"]) > 0
            and int(raw_values["k"]) == raw_expected_k
            and int(power_values["k"]) == power_expected_k
            and raw_status["requested_algorithm"] == "rbc"
            and power_status["requested_algorithm"] == "rbc"
            and raw_status["empirical_audit_passed"]
            and power_status["empirical_audit_passed"]
            and not raw_status["hidden_fallback_allowed"]
            and not power_status["hidden_fallback_allowed"]
        )
    except (KeyError, TypeError, ValueError):
        return False

def acceptance_for(record):
    regularization = record["regularization"]
    power = record["power_queries"]
    accuracy = record["accuracy"]
    resources = record["resources"]
    envelope_bound = record["bound_terms"]["envelope_total"]
    base_bound = record["bound_terms"]["base_total"]
    persistence_max = record["persistence"]["max"]
    gamma_observed = regularization["relative_distortion_ratio_max"]
    unresolved_fraction = regularization["local_scale_below_resolution_fraction"]
    config = record["config"]
    grid_selection = record["grid_selection"]
    actual_grid = record["grid"]
    checks = {
        "all_cpu_tests": bool(CPU_TESTS_OK),
        "gpu_stack_blackwell": bool(GPU_STACK_OK and cc[0] >= 10),
        "rbc_audits_empirical": bool(RBC_AUDITS_OK),
        "run_rbc_audits_match_capacities": record_rbc_audits_pass(record),
        "cubical_msf_gpu_structural": bool(MSF_SMOKE_OK),
        "fused_budget_matches_cpu": bool(FUSED_BUDGET_PARITY_OK),
        "local_budget_cuda_smoke": bool(LOCAL_BUDGET_SMOKE_OK),
        "local_fused_solver": regularization["solver_backend"] == "cuda_fused_budget_v1",
        "local_grid_policy": accuracy["grid_policy"] == "local_scale",
        "spatial_mode_is_enveloped": accuracy["mode"] == "spatial_enveloped",
        "conditional_guard_fraction_is_one": float(power["conditional_guard_pass_fraction"]) == 1.0,
        "uncertain_is_zero": int(power["uncertain_queries"]) == 0,
        "budget_violation": float(regularization["distortion_budget_violation_max"]) <= RESIDUAL_TOL,
        "relative_gamma_observed": gamma_observed is not None and float(gamma_observed) < 0.5 and float(gamma_observed) <= DISTORTION_GAMMA + RESIDUAL_TOL,
        "relative_contract_status_exact": regularization["relative_contract_status"] == "conditional_knn_order_float_enveloped",
        "relative_contract_config": config["regularization_mode"] == "local_distortion" and config["distortion_budget_relative"] is not None and int(config["local_scale_k"]) == int(config["K"]),
        "base_bound_matches": bool(np.isclose(base_bound, accuracy["base_total_interleaving_radius"], rtol=2e-6, atol=2e-7)),
        "envelope_bound_matches": bool(np.isclose(envelope_bound, accuracy["total_interleaving_radius"], rtol=2e-6, atol=2e-7)),
        "spatial_error_at_rmin": float(accuracy["relative_spatial_error_at_min_radius"]) <= MAX_RELATIVE_SPATIAL_ERROR + RESIDUAL_TOL,
        "selected_rmin_matches_run": bool(np.isclose(float(config["min_resolved_radius"]), float(grid_selection["selected_radius"]), rtol=1e-12, atol=0.0) and np.isclose(float(accuracy["min_resolved_radius"]), float(config["min_resolved_radius"]), rtol=1e-12, atol=0.0)),
        "selected_grid_within_budget": int(grid_selection["selected_cells"]) <= int(config["max_grid_cells"]) and int(np.prod(grid_selection["selected_shape"], dtype=np.int64)) == int(grid_selection["selected_cells"]),
        "actual_grid_within_plan": actual_grid["policy"] == "local_scale" and int(np.prod(actual_grid["shape"], dtype=np.int64)) <= int(grid_selection["selected_cells"]),
        "vram_below_limit": float(resources["vram_peak_fraction"]) < MAX_VRAM_FRACTION,
        "host_rss_below_limit": 0 < max(int(resources["rss_peak_bytes"]), int(record["rss_after_save_bytes"])) < MAX_HOST_RAM_FRACTION * int(HARDWARE_MANIFEST_PRE["ram"]["total_bytes"]),
        "scratch_remaining": min(int(resources["disk_free_min_bytes"]), int(record["disk_free_after_save_bytes"])) >= int(MIN_DISK_FREE_GIB * 2**30),
        "resource_monitor_sampled": int(resources["samples"]) >= 2 and int(resources["nvml_total_bytes"]) > 0,
        "K_fixed_to_10": int(record["config"]["K"]) == 10,
        "N_fixed_to_30M": int(record["n_points"]) == N_FULL,
    }
    hard_ok = all(checks.values())
    scale_resolved = unresolved_fraction is not None and float(unresolved_fraction) <= MAX_UNRESOLVED_LOCAL_SCALE_FRACTION
    persistence_resolved = bool(record["persistence"]["count_above_comparison_threshold"] > 0 and persistence_max > 2.0 * envelope_bound)
    status = "FAIL" if not hard_ok else ("CONDITIONAL_PASS" if scale_resolved and persistence_resolved else "INCONCLUSIVE")
    return {
        "run_name": record["run_name"], "status": status, "hard_checks": checks,
        "base_interleaving_bound": base_bound, "envelope_interleaving_bound": envelope_bound,
        "local_scale_below_rmin_fraction": unresolved_fraction, "local_scale_resolved": scale_resolved,
        "persistence_max": persistence_max, "persistence_resolved": persistence_resolved,
        "interpretation": "Contrat H0 conditionnel à l'ordre RBC audité; ni labels plats 30 M, ni consistance asymptotique.",
    }

if FULL_RUN_RECORDS:
    ACCEPTANCE_RUNS = [acceptance_for(record) for record in FULL_RUN_RECORDS]
    statuses = [row["status"] for row in ACCEPTANCE_RUNS]
    OVERALL_STATUS = "FAIL" if "FAIL" in statuses else ("INCONCLUSIVE" if "INCONCLUSIVE" in statuses else "CONDITIONAL_PASS")
else:
    ACCEPTANCE_RUNS = []
    OVERALL_STATUS = "INCONCLUSIVE"

ACCEPTANCE = {
    "schema": "perg_hgp.blackwell_acceptance", "schema_version": 1,
    "git_head": GIT_HEAD, "n_full": N_FULL, "K": K,
    "local_scale_policy": {"target_quantile": LOCAL_SCALE_TARGET_QUANTILE, "dimension_assumption": LOCAL_SCALE_DIMENSION, "manual_override": MIN_RESOLVED_RADIUS_OVERRIDE, "max_grid_cells": MAX_GRID_CELLS},
    "overall_status": OVERALL_STATUS, "full_runs": ACCEPTANCE_RUNS,
    "warnings": [
        "RBC est audité par échantillons contre cuVS brute force, pas prouvé sur 30 M.",
        "La grille est uniforme anisotrope pilotée par r_min, pas encore adaptative par cellule.",
        "Le quantile pilote est extrapolé par N^(-1/d) avec d=3 ; le plancher de budget de grille peut rendre le résultat INCONCLUSIVE.",
        "simplex_residual_max vaut null sur le kernel fusionné car cette métrique n'est pas mesurée ; la parité CPU/GPU est le contrôle du solveur.",
        "K=10 est une résolution finie; aucun label plat 30 M n'est produit.",
    ],
}
with open(ARTIFACT_ROOT / "acceptance.json", "w", encoding="utf-8") as stream:
    json.dump(ACCEPTANCE, stream, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)
    stream.write("\n")
print(json.dumps(ACCEPTANCE, indent=2, ensure_ascii=False))


## 11. Archive locale finale

Chaque run contient au minimum `run_report.json`, `benchmark_metrics.json`, `cubical_hierarchy.npz` et `acceptance_inputs.npz`. La cellule crée une archive locale dans `/content`. L'archive ne doit être copiée vers Drive qu'après la fin de tous les calculs, si une conservation externe est souhaitée.


In [ ]:
import tarfile

archive_stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ARCHIVE_PATH = assert_local_scratch(Path("/content") / f"perg_hgp_blackwell_results_{archive_stamp}.tar.gz")
archive_disk_free_before_bytes = int(shutil.disk_usage("/content").free)
archive_rss_before_bytes = int(psutil.Process().memory_info().rss)
with tarfile.open(ARCHIVE_PATH, "w:gz") as archive:
    archive.add(ARTIFACT_ROOT, arcname=ARTIFACT_ROOT.name)
archive_size_bytes = int(ARCHIVE_PATH.stat().st_size)
archive_disk_free_after_bytes = int(shutil.disk_usage("/content").free)
archive_rss_after_bytes = int(psutil.Process().memory_info().rss)
if archive_size_bytes <= 0:
    raise RuntimeError("L'archive finale est vide.")
if archive_disk_free_after_bytes < int(MIN_DISK_FREE_GIB * 2**30):
    raise OSError("L'archive finale a fait passer le scratch sous le seuil minimal de disque libre.")
ARCHIVE_METRICS = {
    "status": OVERALL_STATUS, "artifact_root": str(ARTIFACT_ROOT),
    "archive": str(ARCHIVE_PATH), "archive_bytes": archive_size_bytes,
    "disk_free_before_bytes": archive_disk_free_before_bytes,
    "disk_free_after_bytes": archive_disk_free_after_bytes,
    "rss_before_bytes": archive_rss_before_bytes, "rss_after_bytes": archive_rss_after_bytes,
}
print(json.dumps(ARCHIVE_METRICS, indent=2, ensure_ascii=False))
